# 03 · WhatsApp chats → sentiment arc per session

The modern version of the legacy customer-service pipeline:

```mermaid
flowchart LR
    exp["WhatsApp export<br/>.txt / .zip"] --> parse["load_whatsapp<br/>parse + split sessions"]
    parse --> media["describe_media<br/>(optional)"]
    media --> anon["anonymize"]
    anon --> arc["run_task<br/>SENTIMENT_ARC"]
    arc --> out["transition table<br/>+ timeline"]
```

**Get an export:** in WhatsApp open the chat → ⋮ / contact name → *Export chat* → *Include media* (zip) or *Without media* (txt). Put it in `data/raw/whatsapp/` (git-ignored). Without an export, this notebook runs on the synthetic fixture in `tests/fixtures/`.

**Privacy:** `anonymize` replaces names with roles or pseudonyms and redacts phones, emails, URLs and IDs *before* the text goes to a model. `describe_media` sends media files **raw**, so enable it only for data you may share with the provider. The name → pseudonym mapping stays in memory; don't save it next to the results.

In [ ]:
%load_ext autoreload
%autoreload 2

import logging
import os

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv

from genaianalysis.ingest.whatsapp import load_whatsapp
from genaianalysis.media import describe_media
from genaianalysis.privacy import anonymize
from genaianalysis.run import run_task
from genaianalysis.tasks.sentiment import SENTIMENT_ARC
from genaianalysis.utils.paths import data_processed_dir, data_raw_dir, project_dir

load_dotenv(project_dir(".env"))
os.environ.setdefault("PYDANTIC_AI_NO_BANNER", "1")
logging.basicConfig(level=logging.ERROR)
pd.set_option("display.max_colwidth", 160)

## Config

In [ ]:
exports = sorted(data_raw_dir("whatsapp").glob("*.zip")) + sorted(data_raw_dir("whatsapp").glob("*.txt"))
EXPORT = exports[0] if exports else project_dir("tests", "fixtures", "whatsapp_android_es.txt")
ROLES = {"Soporte Tienda": "agent"}  # exact sender name -> role; everyone else becomes P1, P2, ...
SESSION_GAP = pd.Timedelta(hours=6)  # silence that starts a new conversation
DESCRIBE_MEDIA = False  # True sends attached images / voice notes to MEDIA_MODEL
MODEL = "google:gemini-flash-lite-latest"
MEDIA_MODEL = "google:gemini-flash-lite-latest"
RPM = 12
EXPORT.name

## 1 · Parse

In [ ]:
sessions = load_whatsapp(EXPORT, session_gap=SESSION_GAP.to_pytimedelta())
overview = pd.DataFrame(
    {
        "id": c.id,
        "start": c.messages[0].timestamp,
        "messages": len(c.messages),
        "media": sum(bool(m.media) for m in c.messages),
        "senders": len({m.sender for m in c.messages}),
    }
    for c in sessions
)
print(f"{len(sessions)} sessions, {overview.messages.sum()} messages")
overview.describe(include="all").T[["count", "mean", "min", "max"]]

## 2 · Media → text (optional)
Runs before `anonymize` so names that appear in transcriptions get pseudonymized too.

In [ ]:
if DESCRIBE_MEDIA:
    sessions = await describe_media(sessions, MEDIA_MODEL, language="Spanish", rpm=RPM)

## 3 · Anonymize

In [ ]:
convs, mapping = anonymize(sessions, roles=ROLES)
print(f"{len(mapping)} senders pseudonymized")
print(convs[0].to_text()[:1500])

## 4 · Sentiment arc per session

In [ ]:
arcs = await run_task(SENTIMENT_ARC, convs, MODEL, rpm=RPM)
arcs = arcs.merge(overview.assign(id=[c.id for c in convs])[["id", "start", "messages"]], on="id")
arcs[["start", "messages", "initial_sentiment", "final_sentiment", "final_sentiment_justification", "error"]]

How sessions move from start to end (rows: initial, columns: final):

In [ ]:
pd.crosstab(arcs["initial_sentiment"], arcs["final_sentiment"], margins=True)

In [ ]:
score = {"negative": -1, "neutral": 0, "positive": 1}
t = arcs.dropna(subset=["final_sentiment"]).sort_values("start")
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t["start"], t["initial_sentiment"].map(score), "o", alpha=0.5, label="initial")
ax.plot(t["start"], t["final_sentiment"].map(score), "s-", label="final")
ax.set(yticks=[-1, 0, 1], yticklabels=["negative", "neutral", "positive"], title="Sentiment per session")
ax.legend();

## 5 · Optional: Jev on the same sessions
Labels only (no justifications), calibrated probabilities, much cheaper. Needs `TYPESAFE_API_KEY`.

In [ ]:
if os.getenv("TYPESAFE_API_KEY"):
    from genaianalysis.backends.jev import run_jev

    jev = await run_jev(SENTIMENT_ARC, convs)
    display(jev[["id", "initial_sentiment", "initial_sentiment_confidence", "final_sentiment", "final_sentiment_confidence"]])

## Save
Anonymized results only (git-ignored).

In [ ]:
arcs.to_parquet(data_processed_dir(f"whatsapp_arcs_{EXPORT.stem.replace(' ', '_')}.parquet"))